In [ ]:
"""
============================================================
Lab Assignment 1 – Neural Network Implementation
Using TensorFlow / Keras
Task   : Multi-class Classification on MNIST (digits 0-9)
Display: Weights, Loss, Accuracy & Gradients per Epoch
============================================================
"""

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────
# REPRODUCIBILITY
# ─────────────────────────────────────────────────────────
np.random.seed(42)
tf.random.set_seed(42)

# ─────────────────────────────────────────────────────────
# 1. LOAD & PRE-PROCESS MNIST DATASET
# ─────────────────────────────────────────────────────────
print("=" * 65)
print("   LAB ASSIGNMENT 1 – Neural Network with TensorFlow/Keras")
print("=" * 65)
print("\n[1] Loading MNIST dataset ...")

(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

# Normalise pixel values to [0, 1]
X_train = X_train.reshape(-1, 784).astype("float32") / 255.0
X_test  = X_test.reshape(-1, 784).astype("float32") / 255.0

# One-hot encode labels  (10 classes: digits 0-9)
y_train_ohe = keras.utils.to_categorical(y_train, num_classes=10)
y_test_ohe  = keras.utils.to_categorical(y_test,  num_classes=10)

print(f"   Training samples : {X_train.shape[0]}")
print(f"   Test samples     : {X_test.shape[0]}")
print(f"   Input features   : {X_train.shape[1]}  (28×28 pixels)")
print(f"   Output classes   : 10  (digits 0–9)")

# ─────────────────────────────────────────────────────────
# 2. BUILD THE MODEL
# ─────────────────────────────────────────────────────────
print("\n[2] Building model architecture ...")

model = keras.Sequential([
    # Hidden Layer 1
    layers.Dense(128, activation="relu",    input_shape=(784,), name="Hidden_1"),
    layers.Dropout(0.2, name="Dropout_1"),

    # Hidden Layer 2
    layers.Dense(64,  activation="relu",    name="Hidden_2"),
    layers.Dropout(0.2, name="Dropout_2"),

    # Output Layer  – softmax for multi-class probabilities
    layers.Dense(10,  activation="softmax", name="Output"),
], name="MNIST_Classifier")

model.summary()

# ─────────────────────────────────────────────────────────
# 3. COMPILE THE MODEL
# ─────────────────────────────────────────────────────────
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# ─────────────────────────────────────────────────────────
# 4. CUSTOM CALLBACK – prints weights, loss, acc & gradients
# ─────────────────────────────────────────────────────────
class EpochDetailCallback(keras.callbacks.Callback):
    """Logs weights, gradients, loss and accuracy after every epoch."""

    def __init__(self, X_batch, y_batch, max_weights_display=5):
        super().__init__()
        # Keep a small fixed batch to compute gradients on
        self.X_batch = tf.constant(X_batch[:64])
        self.y_batch = tf.constant(y_batch[:64])
        self.max_w   = max_weights_display   # first N weights to show per layer

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        print("\n" + "─" * 65)
        print(f"  EPOCH {epoch + 1:03d}  │  "
              f"Loss: {logs.get('loss', 0):.6f}  │  "
              f"Accuracy: {logs.get('accuracy', 0)*100:.2f}%  │  "
              f"Val-Loss: {logs.get('val_loss', 0):.6f}  │  "
              f"Val-Acc: {logs.get('val_accuracy', 0)*100:.2f}%")
        print("─" * 65)

        # ── Weights per layer ──────────────────────────────────────
        print("  [WEIGHTS UPDATED]")
        for layer in self.model.layers:
            weights = layer.get_weights()
            if not weights:
                continue              # skip layers without weights (Dropout)
            W, b = weights[0], weights[1]
            flat_W = W.flatten()
            displayed = flat_W[:self.max_w]
            print(f"    Layer '{layer.name}'")
            print(f"      Shape   : W={W.shape}, b={b.shape}")
            print(f"      W[:5]   : {np.round(displayed, 6)}")
            print(f"      W  mean : {flat_W.mean():.6f}  |  "
                  f"std: {flat_W.std():.6f}  |  "
                  f"min: {flat_W.min():.6f}  |  "
                  f"max: {flat_W.max():.6f}")
            print(f"      b[:5]   : {np.round(b[:self.max_w], 6)}")

        # ── Gradients per layer ────────────────────────────────────
        print("\n  [GRADIENTS]")
        loss_fn = keras.losses.CategoricalCrossentropy()
        with tf.GradientTape() as tape:
            preds = self.model(self.X_batch, training=False)
            loss  = loss_fn(self.y_batch, preds)

        trainable_vars = self.model.trainable_variables
        gradients      = tape.gradient(loss, trainable_vars)

        for var, grad in zip(trainable_vars, gradients):
            if grad is None:
                continue
            g = grad.numpy().flatten()
            print(f"    Grad '{var.name}'")
            print(f"      grad[:5] : {np.round(g[:self.max_w], 8)}")
            print(f"      mean: {g.mean():.8f}  |  "
                  f"std: {g.std():.8f}  |  "
                  f"norm: {np.linalg.norm(g):.8f}")

        print()


# ─────────────────────────────────────────────────────────
# 5. TRAIN THE MODEL
# ─────────────────────────────────────────────────────────
EPOCHS     = 10
BATCH_SIZE = 128

print("\n[3] Training started ...\n")

callback = EpochDetailCallback(X_train, y_train_ohe, max_weights_display=5)

history = model.fit(
    X_train, y_train_ohe,
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    validation_split= 0.1,        # 10 % of training data used for validation
    callbacks       = [callback],
    verbose         = 0           # suppress default output – we print our own
)

# ─────────────────────────────────────────────────────────
# 6. EVALUATE ON TEST SET
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("[4] Final Evaluation on Test Set")
print("=" * 65)

test_loss, test_acc = model.evaluate(X_test, y_test_ohe, verbose=0)
print(f"   Test Loss     : {test_loss:.6f}")
print(f"   Test Accuracy : {test_acc * 100:.2f}%")

# ─────────────────────────────────────────────────────────
# 7. SAMPLE PREDICTIONS
# ─────────────────────────────────────────────────────────
print("\n[5] Sample Predictions (first 10 test images)")
print("─" * 45)
print(f"  {'Index':<7} {'True Label':<13} {'Predicted':<12} {'Confidence'}")
print("─" * 45)

preds      = model.predict(X_test[:10], verbose=0)
pred_class = np.argmax(preds, axis=1)

for i in range(10):
    conf = preds[i][pred_class[i]] * 100
    match = "✓" if pred_class[i] == y_test[i] else "✗"
    print(f"  {i:<7} {y_test[i]:<13} {pred_class[i]:<12} {conf:.2f}%  {match}")

# ─────────────────────────────────────────────────────────
# 8. TRAINING SUMMARY TABLE
# ─────────────────────────────────────────────────────────
print("\n[6] Training History Summary")
print("─" * 65)
print(f"  {'Epoch':<8} {'Train Loss':<14} {'Train Acc':<14} {'Val Loss':<14} {'Val Acc'}")
print("─" * 65)

for ep in range(EPOCHS):
    tl  = history.history["loss"][ep]
    ta  = history.history["accuracy"][ep] * 100
    vl  = history.history["val_loss"][ep]
    va  = history.history["val_accuracy"][ep] * 100
    print(f"  {ep+1:<8} {tl:<14.6f} {ta:<14.2f} {vl:<14.6f} {va:.2f}%")

print("─" * 65)
print("\n  [DONE] Training complete.\n")

# ─────────────────────────────────────────────────────────
# 9. SAVE THE MODEL
# ─────────────────────────────────────────────────────────
model.save("mnist_neural_network.keras")
print("  Model saved as  →  mnist_neural_network.keras")
print("=" * 65)


   LAB ASSIGNMENT 1 – Neural Network with TensorFlow/Keras

[1] Loading MNIST dataset ...
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
   Training samples : 60000
   Test samples     : 10000
   Input features   : 784  (28×28 pixels)
   Output classes   : 10  (digits 0–9)

[2] Building model architecture ...


Model: "MNIST_Classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Hidden_1 (Dense)                │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Hidden_2 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output (Dense)                  │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,386 (427.29 KB)

 Trainable params: 109,386 (427.29 KB)

 Non-trainable params: 0 (0.00 B)


[3] Training started ...


─────────────────────────────────────────────────────────────────
  EPOCH 001  │  Loss: 0.474941  │  Accuracy: 85.92%  │  Val-Loss: 0.150733  │  Val-Acc: 95.83%
─────────────────────────────────────────────────────────────────
  [WEIGHTS UPDATED]
    Layer 'Hidden_1'
      Shape   : W=(784, 128), b=(128,)
      W[:5]   : [ 0.054674 -0.001295  0.056995  0.058859  0.035033]
      W  mean : -0.001990  |  std: 0.054863  |  min: -0.216279  |  max: 0.228903
      b[:5]   : [ 0.022537  0.04159   0.047472 -0.006409  0.011163]
    Layer 'Hidden_2'
      Shape   : W=(128, 64), b=(64,)
      W[:5]   : [-0.120658  0.000732  0.135124 -0.120041  0.021071]
      W  mean : 0.009652  |  std: 0.116404  |  min: -0.311555  |  max: 0.361326
      b[:5]   : [-0.008116  0.039928 -0.00243  -0.000167 -0.042103]
    Layer 'Output'
      Shape   : W=(64, 10), b=(10,)
      W[:5]   : [-0.064212 -0.006821  0.187477  0.09987   0.100397]
      W  mean : -0.011583  |  std: 0.187075  |  min